In [1]:
import os
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from recovar_torch.config import BATCH_SIZE
from recovar_torch.representation_learning_models import (
    RepresentationLearningSingleAutoencoder,
    RepresentationLearningDenoisingSingleAutoencoder,
    RepresentationLearningMultipleAutoencoder,
)

In [2]:
TRAIN_DATA_PATH = "data/X_train_1280sample.npy"
TEST_DATA_PATH = "data/X_test_1280sample.npy"
TRAIN_LABEL_PATH = "data/Y_train_1280sample.npy"
TEST_LABEL_PATH = "data/Y_test_1280sample.npy"

CHECKPOINT_DIR = "checkpoints"
MODEL_SAVE_PATH = "checkpoints/representation_cross_covariances.pt"

EPOCHS = int(os.environ.get("RECOVAR_EPOCHS", 2))
LEARNING_RATE = 1e-3
VAL_SPLIT = 0.1
PATIENCE = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

X_train = np.load(TRAIN_DATA_PATH).astype(np.float32)
print(f"Training data shape: {X_train.shape}")

Training data shape: (1280, 3000, 3)


In [4]:
# model = RepresentationLearningSingleAutoencoder()
# model = RepresentationLearningDenoisingSingleAutoencoder(
#     input_noise_std=1e-6, denoising_noise_std=2e-1)
model = RepresentationLearningMultipleAutoencoder(input_noise_std=1e-6, eps=1e-27).to(DEVICE)

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [6]:
x = torch.from_numpy(X_train)
n_val = int(len(x) * VAL_SPLIT)
n_train = len(x) - n_val
x_tr, x_val = x[:n_train], x[n_train:]

train_loader = DataLoader(TensorDataset(x_tr), batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
val_loader = DataLoader(TensorDataset(x_val), batch_size=BATCH_SIZE, shuffle=False, drop_last=True)


def run_epoch(loader, train):
    model.train(train)
    total, n = 0.0, 0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            out = model(xb)
            loss = out[-1]
        if train:
            loss.backward()
            optimizer.step()
        total += float(loss) * xb.shape[0]
        n += xb.shape[0]
    return total / max(n, 1)

In [7]:
best_val = float("inf")
best_state = None
wait = 0
history = {"loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    tr_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False) if len(val_loader) > 0 else tr_loss

    history["loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS}  loss={tr_loss:.4f}  val_loss={val_loss:.4f}")

    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"autoencoder_epoch_{epoch + 1:02d}.pt"))

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

Epoch 01/2  loss=2.7143  val_loss=2.7143
Epoch 02/2  loss=2.5318  val_loss=2.5318
Model saved to checkpoints/representation_cross_covariances.pt
